# Bloyce's Protocol — Strict Complaint Processor
## NormalObjects Lab 2 | LangGraph Structured Workflow

This notebook builds a **rule-based, step-enforced** complaint processor for the Downside Up universe.  
Unlike Lab 1 (LangChain agent that chose its own path), every complaint here **must** pass through:

```
INTAKE → VALIDATE → INVESTIGATE → RESOLVE → CLOSE
```

No step can be skipped. The graph structure enforces the order.

---
### What you'll build
| Step | What happens |
|---|---|
| **Intake** | Categorize the complaint (portal / monster / psychic / environmental / other) |
| **Validate** | Check it meets category-specific rules — reject if not |
| **Investigate** | Gather evidence relevant to the category |
| **Resolve** | Propose a fix with an effectiveness rating |
| **Close** | Log confirmation, satisfaction check, timestamp |

## Step 1 — Install packages

Run this once to install everything needed.

In [ ]:
# Install all required libraries
# langchain-openai  → connects LangChain to the OpenAI API
# langgraph         → the state machine / workflow engine
# python-dotenv     → loads the API key from .env
%pip install -q langchain langchain-openai langgraph python-dotenv

## Step 2 — Imports and LLM setup

`temperature=0` means the LLM always picks the most likely answer — no randomness.  
This is important for a rules-based system that needs consistent results.

In [ ]:
import os
import sys
from datetime import datetime
from typing import TypedDict, List, Optional
from dotenv import load_dotenv

# Fix Windows terminal encoding
if sys.stdout.encoding and sys.stdout.encoding.lower() != "utf-8":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END

# Load OPENAI_API_KEY from .env file
load_dotenv()

# temperature=0 → no randomness, same input always gives same output
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✓ LLM ready")

## Step 3 — Define the State

The **state** is a dictionary that travels through every node in the graph.  
Each node reads from it and writes an updated version back.  
Think of it like a form that gets filled in one section at a time.

```
ComplaintState
├── complaint          ← the original text (never changes)
├── category           ← filled in by INTAKE
├── is_valid           ← filled in by VALIDATE
├── rejection_reason   ← filled in by VALIDATE (if rejected)
├── investigation      ← filled in by INVESTIGATE
├── resolution         ← filled in by RESOLVE
├── effectiveness      ← filled in by RESOLVE
├── closure_confirmed  ← filled in by CLOSE
├── workflow_path      ← every node appends its name here
└── status             ← updated at each step
```

In [ ]:
# TypedDict = a dictionary with type hints
# Optional[str] = the field can be a string OR None (not filled yet)

class ComplaintState(TypedDict):
    complaint: str                   # original complaint text
    category: Optional[str]          # portal | monster | psychic | environmental | other
    is_valid: Optional[bool]         # True = passes validation
    rejection_reason: Optional[str]  # set when is_valid is False
    investigation: Optional[str]     # evidence report from the investigate node
    resolution: Optional[str]        # proposed fix
    effectiveness: Optional[str]     # high | medium | low
    closure_confirmed: Optional[bool]
    satisfaction_checked: Optional[bool]
    workflow_path: List[str]         # audit trail — every node name appended here
    status: str                      # current step name
    timestamp: Optional[str]         # set at closure

# The 5 valid categories (used by intake and validate)
VALID_CATEGORIES = {"portal", "monster", "psychic", "environmental", "other"}

print("✓ State structure defined")

## Step 4 — Node 1: INTAKE

**What it does:** reads the complaint and assigns a category.  
The LLM is asked to return exactly one word from the allowed list.  
If the response isn't one of the 5 valid categories, we fall back to `"other"`.

In [ ]:
def intake_node(state: ComplaintState) -> ComplaintState:
    print("\n[INTAKE] Processing complaint...")

    complaint = state["complaint"]

    # Ask the LLM to categorize — we want exactly one word back
    prompt = f"""Categorize this Downside Up complaint into exactly one of these categories:
- portal: Issues with portal timing, location, or behavior
- monster: Issues with creature behavior (demogorgons, etc.)
- psychic: Issues with psychic abilities or limitations
- environmental: Issues with electricity, weather, or physical environment
- other: Anything else

Complaint: {complaint}

Reply with ONLY the category name (one word, lowercase)."""

    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip().lower()

    # Safety net: if LLM returned something unexpected, use "other"
    category = raw if raw in VALID_CATEGORIES else "other"

    print(f"[INTAKE] Categorized as: {category}")

    # Return the full state with only the new fields updated
    # workflow_path grows with each node — it's the audit trail
    return {
        **state,
        "category": category,
        "workflow_path": state.get("workflow_path", []) + ["intake"],
        "status": "intake",
    }

print("✓ intake_node defined")

## Step 5 — Node 2: VALIDATE

**What it does:** checks whether the complaint has enough detail to process.  
Each category has a specific rule from Bloyce's Protocol.  
`"other"` complaints are always rejected (auto-escalated to manual review).  

The result sets `is_valid = True` or `False`, which the router node reads next.

In [ ]:
def validate_node(state: ComplaintState) -> ComplaintState:
    print("\n[VALIDATE] Checking complaint against Bloyce's Protocol...")

    category = state["category"]
    complaint = state["complaint"]

    # "other" always fails — no rule to check against, manual review required
    if category == "other":
        print("[VALIDATE] Category 'other' — auto-escalated for manual review.")
        return {
            **state,
            "is_valid": False,
            "rejection_reason": "Category 'other' requires manual review escalation per Bloyce's Protocol.",
            "workflow_path": state["workflow_path"] + ["validate"],
            "status": "validate",
        }

    # Each category has a specific rule that must be satisfied
    rules = {
        "portal":        "Must reference a specific location or timing anomaly.",
        "monster":       "Must describe creature behavior or a specific interaction.",
        "psychic":       "Must reference a specific ability limitation or malfunction.",
        "environmental": "Must mention electricity, weather, or an observable physical phenomenon.",
    }

    prompt = f"""You are validating a Downside Up complaint under Bloyce's Protocol.

Category: {category}
Validation rule: {rules[category]}
Complaint: {complaint}

Does the complaint satisfy the rule above?
Reply with exactly: VALID or INVALID
Then on a new line explain in one sentence why."""

    response = llm.invoke([HumanMessage(content=prompt)])
    lines = response.content.strip().splitlines()
    verdict = lines[0].strip().upper()
    reason = lines[1].strip() if len(lines) > 1 else "No reason provided."

    is_valid = verdict == "VALID"
    rejection_reason = None if is_valid else reason

    print(f"[VALIDATE] Result: {verdict} — {reason}")

    return {
        **state,
        "is_valid": is_valid,
        "rejection_reason": rejection_reason,
        "workflow_path": state["workflow_path"] + ["validate"],
        "status": "validate",
    }

print("✓ validate_node defined")

## Step 6 — Routing function + REJECT node

**Routing function:** after `validate`, LangGraph calls this function to decide what comes next.  
It reads `is_valid` from the state and returns either `"investigate"` or `"reject"`.

**Reject node:** the dead-end branch — explains why the complaint was rejected and stops.

In [ ]:
# Routing function — called by LangGraph after validate
# Must return a string that matches one of the keys in add_conditional_edges
def route_after_validate(state: ComplaintState) -> str:
    if state["is_valid"]:
        return "investigate"  # valid → continue the happy path
    return "reject"           # invalid → dead-end branch


# Reject node — only reached when validation fails
def reject_node(state: ComplaintState) -> ComplaintState:
    print("\n[REJECT] Complaint did not meet validation requirements.")
    print(f"[REJECT] Reason: {state['rejection_reason']}")

    return {
        **state,
        "workflow_path": state["workflow_path"] + ["reject"],
        "status": "rejected",
    }

print("✓ router and reject_node defined")

## Step 7 — Node 3: INVESTIGATE

**What it does:** gathers evidence specific to the complaint category.  
This node can only be reached if validation passed — the graph edge enforces this.  
The LLM produces a short investigation report that the next node (resolve) will use.

In [ ]:
def investigate_node(state: ComplaintState) -> ComplaintState:
    print("\n[INVESTIGATE] Gathering evidence...")

    category = state["category"]
    complaint = state["complaint"]

    # Each category has a different investigation angle per Bloyce's Protocol
    focus = {
        "portal":        "temporal patterns, location consistency, and environmental factors",
        "monster":       "behavioral data, interaction patterns, and environmental triggers",
        "psychic":       "ability specifications, tested limitations, and contextual factors",
        "environmental": "power line activity, atmospheric conditions, and anomaly correlation",
    }

    prompt = f"""You are an investigator at the Downside Up Complaint Bureau.

Category: {category}
Investigation focus: {focus.get(category, 'general anomaly patterns')}
Complaint: {complaint}

Produce a short investigation report (3-5 sentences) documenting your findings.
Reference specific evidence and patterns you observe."""

    response = llm.invoke([HumanMessage(content=prompt)])
    investigation = response.content.strip()

    print(f"[INVESTIGATE] Evidence documented.")

    return {
        **state,
        "investigation": investigation,
        "workflow_path": state["workflow_path"] + ["investigate"],
        "status": "investigate",
    }

print("✓ investigate_node defined")

## Step 8 — Node 4: RESOLVE

**What it does:** proposes a fix using the investigation findings.  
The resolution must reference an established Downside Up procedure.  
It must also include an effectiveness rating: `high`, `medium`, or `low`.  
Low-rated resolutions will trigger a 30-day follow-up note at closure.

In [ ]:
def resolve_node(state: ComplaintState) -> ComplaintState:
    print("\n[RESOLVE] Determining resolution...")

    prompt = f"""You are resolving a Downside Up complaint under Bloyce's Protocol.

Category: {state['category']}
Original complaint: {state['complaint']}
Investigation findings: {state['investigation']}

Provide:
1. A specific resolution that references an established Downside Up procedure or protocol.
2. An effectiveness rating: high, medium, or low (with one-sentence justification).

Format your response as:
RESOLUTION: <your resolution here>
EFFECTIVENESS: <high|medium|low> — <one sentence reason>"""

    response = llm.invoke([HumanMessage(content=prompt)])
    text = response.content.strip()

    # Parse the two structured lines out of the response
    resolution = ""
    effectiveness = "medium"  # safe fallback if parsing fails
    for line in text.splitlines():
        if line.upper().startswith("RESOLUTION:"):
            resolution = line.split(":", 1)[1].strip()
        elif line.upper().startswith("EFFECTIVENESS:"):
            raw_eff = line.split(":", 1)[1].strip().lower()
            for level in ("high", "medium", "low"):
                if level in raw_eff:
                    effectiveness = level
                    break

    print(f"[RESOLVE] Resolution found. Effectiveness: {effectiveness}")

    return {
        **state,
        "resolution": resolution or text,  # fallback: full text if parsing failed
        "effectiveness": effectiveness,
        "workflow_path": state["workflow_path"] + ["resolve"],
        "status": "resolve",
    }

print("✓ resolve_node defined")

## Step 9 — Node 5: CLOSE

**What it does:** confirms the resolution was applied, records a timestamp, and checks satisfaction.  
If the resolution had a `low` effectiveness rating, a 30-day follow-up checkpoint is flagged.  
This is the last node in the happy path — it connects directly to `END`.

In [ ]:
def close_node(state: ComplaintState) -> ComplaintState:
    print("\n[CLOSE] Closing complaint...")

    # Per protocol: satisfaction must be attempted before closure
    satisfaction_checked = True
    closure_confirmed = True
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Low-effectiveness complaints need a 30-day follow-up
    followup_note = ""
    if state.get("effectiveness") == "low":
        followup_note = " ⚠ 30-day follow-up checkpoint required (low effectiveness rating)."

    print(f"[CLOSE] Complaint closed at {timestamp}.{followup_note}")

    return {
        **state,
        "closure_confirmed": closure_confirmed,
        "satisfaction_checked": satisfaction_checked,
        "timestamp": timestamp,
        "workflow_path": state["workflow_path"] + ["close"],
        "status": "closed",
    }

print("✓ close_node defined")

## Step 10 — Build and Compile the Graph

Now we wire everything together.  

- `add_node` registers a node by name
- `add_edge` creates a fixed connection: A always goes to B
- `add_conditional_edges` creates a branch: call the router function and go to whatever it returns
- `compile()` locks the graph into a callable app

```
intake
  ↓  (always)
validate
  ↓ is_valid=True      ↓ is_valid=False
investigate           reject → END
  ↓  (always)
resolve
  ↓  (always)
close → END
```

In [ ]:
# Create the graph — tells LangGraph what our state looks like
workflow = StateGraph(ComplaintState)

# Register every node (name → function)
workflow.add_node("intake",      intake_node)
workflow.add_node("validate",    validate_node)
workflow.add_node("investigate", investigate_node)
workflow.add_node("resolve",     resolve_node)
workflow.add_node("close",       close_node)
workflow.add_node("reject",      reject_node)

# Entry point — where every complaint starts
workflow.set_entry_point("intake")

# Fixed edges (no branching)
workflow.add_edge("intake",      "validate")   # intake always goes to validate
workflow.add_edge("investigate", "resolve")    # investigate always goes to resolve
workflow.add_edge("resolve",     "close")      # resolve always goes to close
workflow.add_edge("close",       END)          # close finishes the workflow
workflow.add_edge("reject",      END)          # reject is also a terminal node

# Conditional edge after validate — calls route_after_validate to pick the next node
workflow.add_conditional_edges(
    "validate",
    route_after_validate,
    {
        "investigate": "investigate",  # router returns "investigate" → go to investigate node
        "reject":      "reject",        # router returns "reject" → go to reject node
    },
)

# Compile turns the graph definition into a runnable app
app = workflow.compile()

print("✓ Graph compiled — workflow is ready")

## Step 11 — Visualization helper

This function prints a clean summary of what happened after a complaint is processed.  
The `workflow_path` list is the audit trail — every node name was appended in order.

In [ ]:
def print_workflow_result(complaint: str, result: ComplaintState) -> None:
    print("\n" + "=" * 60)
    print(f"COMPLAINT : {complaint[:80]}..." if len(complaint) > 80 else f"COMPLAINT : {complaint}")
    print(f"PATH      : {' → '.join(result['workflow_path'])}")  # the full audit trail
    print(f"STATUS    : {result['status']}")
    print(f"CATEGORY  : {result.get('category', 'n/a')}")

    if result["status"] == "rejected":
        print(f"REJECTED  : {result.get('rejection_reason', 'n/a')}")

    elif result["status"] == "closed":
        print(f"RESOLUTION: {result.get('resolution', 'n/a')}")
        print(f"EFFECTIVE : {result.get('effectiveness', 'n/a')}")
        print(f"CLOSED AT : {result.get('timestamp', 'n/a')}")
        if result.get("effectiveness") == "low":
            print("FOLLOWUP  : ⚠ 30-day checkpoint required")

    print("=" * 60)

print("✓ Visualization helper ready")

## Step 12 — Run the workflow

We test 5 complaints:
1. **Portal** — valid, has location + timing detail
2. **Monster** — valid, describes specific behavior
3. **Psychic** — valid, names a specific limitation
4. **Environmental** — valid, mentions electricity + phenomenon
5. **Vague complaint** — should be rejected (no detail, likely categorized as "other")

Each complaint starts as an `initial_state` with only `complaint` filled in.  
After `app.invoke()` the state has all fields populated by the nodes that ran.

In [ ]:
test_complaints = [
    # 1. Portal — valid
    "The Downside Up portal opens at different times each day near the old Hawkins Lab. How do I predict when it will appear?",
    # 2. Monster — valid
    "Demogorgons sometimes work together in hunting packs and sometimes fight each other. What determines their coordination?",
    # 3. Psychic — valid
    "El can move objects with her mind but cannot lift anything heavier than 50kg. Why is there a weight limit on telekinesis?",
    # 4. Environmental — valid
    "Every time the portal opens, the power lines within 200 metres flicker and surge. Why do creatures and power lines react so strangely together?",
    # 5. Invalid — vague, no category detail
    "This is not a valid complaint about something random",
]

results = []

for complaint in test_complaints:
    # Build the initial state — only complaint is set, everything else is blank
    initial_state: ComplaintState = {
        "complaint": complaint,
        "category": None,
        "is_valid": None,
        "rejection_reason": None,
        "investigation": None,
        "resolution": None,
        "effectiveness": None,
        "closure_confirmed": None,
        "satisfaction_checked": None,
        "workflow_path": [],
        "status": "pending",
        "timestamp": None,
    }

    # LangGraph runs the nodes in order and returns the final state
    final_state = app.invoke(initial_state)
    results.append(final_state)
    print_workflow_result(complaint, final_state)

## Step 13 — Summary table

A quick count of outcomes across all complaints.

In [ ]:
closed   = sum(1 for r in results if r["status"] == "closed")
rejected = sum(1 for r in results if r["status"] == "rejected")

print("\n" + "=" * 60)
print("WORKFLOW EXECUTION SUMMARY")
print("=" * 60)
print(f"Total complaints processed : {len(results)}")
print(f"Successfully closed        : {closed}")
print(f"Rejected                   : {rejected}")
print(f"\nAudit trails:")
for i, r in enumerate(results, 1):
    path = " → ".join(r["workflow_path"])
    print(f"  [{i}] {path}")
print("=" * 60)

## Recap — What makes LangGraph different?

| | LangChain (Lab 1, Week 3) | LangGraph (this lab) |
|---|---|---|
| **Workflow shape** | Agent decides freely | You define every edge in advance |
| **Step order** | Emergent (LLM chooses) | Fixed and enforced by the graph |
| **Can skip steps?** | Yes | No — graph prevents it |
| **Audit trail** | None (just messages) | `workflow_path` tracks every node |
| **Good for** | Creative, exploratory tasks | Compliance, traceability, strict rules |

The key insight: **LangGraph trades flexibility for predictability.**  
When you need to guarantee that investigation always happens before resolution, use LangGraph.  
When you want the AI to figure out its own path, use a LangChain agent.